# WBSNet — Option-A Paper Run on Colab

End-to-end runner for the **Option-A paper scope**:

- 7 ablation variants (A1-A7) × seeds on **Kvasir-SEG**
- Full WBSNet on **CVC-ClinicDB** and **ISIC2018**
- U-Net baselines on Kvasir, ClinicDB, ISIC2018
- Generalization eval: Kvasir checkpoint → CVC-ColonDB
- Aggregation, significance tests, complexity, qualitative figures

**Recommended runtime:** Colab Pro+ A100 with background execution.
**Backup:** Colab Pro L4 for smoke tests or smaller runs.

Before running:
1. `Runtime → Change runtime type → A100 GPU` (or L4 if on Pro).
2. `Runtime → Background execution` (Pro+ only) — lets you close the laptop.
3. Have the processed dataset on Google Drive at `MyDrive/WBSNet_Dataset/` with subdirs `kvasir/`, `cvc_clinicdb/`, `cvc_colondb/`, `isic2018/`.
4. If continuing from the Kaggle seed-3407 run, keep the exported `wbsnet_paper_runs/` folder at `MyDrive/wbsnet_paper_runs/`.
5. The expected processed layout is split-based: `kvasir/train/images`, `kvasir/val/masks`, `isic2018/test/images`, `cvc_colondb/test/masks`, etc.

Each section is independent — re-run only what you need.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Environment check

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Switch runtime to A100 (Pro+) or L4 (Pro).")

gpu_name = torch.cuda.get_device_name(0)
print(f"\nDetected GPU: {gpu_name}")
SUPPORTED = ("A100", "L4", "H100")
if not any(tag in gpu_name for tag in SUPPORTED):
    raise RuntimeError(
        f"Unsupported GPU '{gpu_name}'. The paper pipeline targets A100/L4/H100. "
        "Change via Runtime -> Change runtime type before running the paper pipeline."
    )


## 2. Clone repo

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MrArrogant2002/WBSNet-research-paper.git"
REPO_DIR = Path("/content/WBSNet-research-paper")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin", "main"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)

print("CWD:", os.getcwd())
print("Runtime repo commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)


## 3. Install dependencies

Colab already ships with CUDA-enabled PyTorch. Install the Colab-compatible dependency set, then install this repo with `--no-deps` so editable install does not downgrade the runtime image.

In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-enabled PyTorch. The second install uses --no-deps so pip does not downgrade torch.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)

import torch
print("torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 4. Verify repo structure

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/verify_repo.py"], check=True)


## 5. Mount Google Drive and link dataset

If your dataset is laid out differently, edit `DRIVE_DATASET` and `DATASET_MAP` below.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

DRIVE_DATASET = Path("/content/drive/MyDrive/WBSNet_Dataset")
assert DRIVE_DATASET.exists(), f"Dataset not found at {DRIVE_DATASET}. Edit DRIVE_DATASET in this cell."

DATA_ROOT = REPO_DIR / "data"
DATA_ROOT.mkdir(exist_ok=True)

DATASET_MAP = {
    "kvasir": "Kvasir-SEG",
    "cvc_clinicdb": "CVC-ClinicDB",
    "cvc_colondb": "CVC-ColonDB",
    "isic2018": "ISIC2018",
}

for src_name, dst_name in DATASET_MAP.items():
    src = DRIVE_DATASET / src_name
    dst = DATA_ROOT / dst_name
    if not src.exists():
        print(f"MISSING: {src}")
        continue
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        print(f"EXISTS: {dst} is not a symlink; leaving it unchanged")
        continue
    os.symlink(src, dst)
    print(f"LINKED: {src} -> {dst}")

print("\nDataset directory links:")
for item in sorted(DATA_ROOT.iterdir()):
    if item.is_symlink():
        print(f"{item} -> {item.resolve()}")


## 6. Restore previous outputs and import legacy / Kaggle-session artifacts

Run this **before** the paper pipeline. It pulls every prior result that lives on Drive into `outputs/` so already-completed runs are skipped instead of retrained:

| Drive root | What it holds |
|---|---|
| `wbsnet_paper_runs/` | Original Kaggle seed-3407 — Kvasir A1–A7, CVC-ClinicDB A1–A7, ISIC2018 A1 |
| `wbsnet_kaggle_session1/` | Kaggle Session 1 — ISIC2018 A2 seed 3407 |
| `wbsnet_kaggle_session2/` | Kaggle Session 2 — Kvasir A2/A5/A6/A7 seed 3408 |
| `wbsnet_kaggle_session3/` | Kaggle Session 3 — Kvasir A1/A3/A4 + ClinicDB A1/A2 seed 3408 |
| `WBSNet_outputs/` | Any partial Colab run from a previous session |

See `kaggle-session-plan.md` in the repo root for the strategy behind the three Kaggle sessions and what work remains for this Colab notebook to handle.

In [ ]:
from pathlib import Path
import subprocess
import sys

DRIVE_OUTPUTS = Path("/content/drive/MyDrive/WBSNet_outputs")

# All Drive roots that may hold importable runs in the legacy
# `paper_suite/<dataset>/<variant>/seed_<seed>/<run_name>/` layout.
# Order matters: later entries can overwrite earlier ones if they contain
# the same run, so list newer / higher-quality sources later.
LEGACY_ROOTS = [
    Path("/content/drive/MyDrive/wbsnet_paper_runs"),       # original Kaggle seed-3407
    Path("/content/drive/MyDrive/wbsnet_kaggle_session1"),  # ISIC2018 A2 seed 3407
    Path("/content/drive/MyDrive/wbsnet_kaggle_session2"),  # Kvasir seed 3408 subset
    Path("/content/drive/MyDrive/wbsnet_kaggle_session3"),  # Kvasir/ClinicDB seed 3408 subset
]

if DRIVE_OUTPUTS.exists():
    print(f"Restoring previous Colab outputs from {DRIVE_OUTPUTS}")
    Path("outputs").mkdir(exist_ok=True)
    subprocess.run(["rsync", "-a", "--info=progress2", f"{DRIVE_OUTPUTS}/", "outputs/"], check=True)
else:
    print(f"No previous Colab output backup found at {DRIVE_OUTPUTS}")

imported_any = False
for legacy_root in LEGACY_ROOTS:
    if not legacy_root.exists():
        print(f"  skip: {legacy_root} not present on Drive")
        continue
    print(f"\nImporting from {legacy_root}")
    subprocess.run([
        sys.executable,
        "scripts/import_legacy_paper_runs.py",
        "--legacy-root", str(legacy_root),
        "--output-root", "outputs",
        "--verify-forward",
    ], check=True)
    imported_any = True

if not imported_any:
    print("\nNo legacy / Kaggle-session folders found on Drive; starting from scratch.")

print("\nExisting best checkpoints that will be skipped by the paper runner:")
checkpoints = sorted(Path("outputs").glob("**/checkpoints/best.pt")) if Path("outputs").exists() else []
for ckpt in checkpoints[:120]:
    print(ckpt)
print(f"Total best checkpoints found: {len(checkpoints)}")


## 7. (Optional) W&B login

Skip this cell to run with W&B offline. Set `WANDB_API_KEY` to log online.

In [ ]:
import os

# Preferred: store the key once via the key icon in the left sidebar (Colab Secrets)
# as 'WANDB_API_KEY', then load it here. Falls back gracefully if the secret is absent.
try:
    from google.colab import userdata  # type: ignore[import-not-found]

    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        print("WANDB_API_KEY loaded from Colab Secrets.")
    else:
        print("Colab Secret 'WANDB_API_KEY' not set; W&B will run offline unless you set it below.")
except Exception as exc:  # noqa: BLE001
    print(f"Colab userdata unavailable ({exc}); W&B will run offline unless you set it below.")

# Alternative paths (uncomment one if you do not want to use Colab Secrets):
#  - Hardcode for a single session (NOT recommended; never commit this):
#       os.environ["WANDB_API_KEY"] = "YOUR_WANDB_KEY_HERE"
#  - Place a `.env` file at the repo root with WANDB_API_KEY=...; ExperimentLogger auto-loads it.
#  - Interactive login that does not echo the key:
#       from getpass import getpass; os.environ["WANDB_API_KEY"] = getpass("WANDB key: ")
# Then, to actually log in to the W&B CLI for this session:
# !wandb login --relogin


## 8. Smoke test (1 epoch, batch 2)

Confirms the pipeline runs end-to-end. ~2 min on A100, ~3 min on L4.

In [ ]:
import subprocess
import sys

# Smoke test: 1 epoch, tiny batch. Run this before the full paper run.
cmd = [
    sys.executable, "train.py",
    "--config", "configs/kvasir_wbsnet.yaml",
    "--override",
    "train.epochs=1",
    "train.batch_size=2",
    "train.grad_accum_steps=1",
    "dataset.split_strategy=pre_split_dirs",
    "dataset.num_workers=2",
    "dataset.prefetch_factor=2",
    "evaluation.compute_hd95=false",
    "evaluation.max_visualizations=0",
    "runtime.device=cuda",
    "runtime.wandb.mode=offline",
    "experiment.name=smoke_test",
    "experiment.run_name=smoke_test",
]
subprocess.run(cmd, check=True)


## 9. Run only the remaining Colab A100 jobs

This is the default paper-run cell for the current budget plan. It assumes the Kaggle/legacy outputs from Section 6 have already been imported.

Based on `kaggle-session-plan.md`, Colab should only fill the remaining ISIC2018 seed-3408 gap:

| Seed | Dataset | Run | Config |
|---|---|---|---|
| 3408 | ISIC2018 | A1 baseline / U-Net | `configs/isic2018_unet_baseline.yaml` |
| 3408 | ISIC2018 | A2 full WBSNet | `configs/isic2018_wbsnet.yaml` |

The cell is idempotent: if `checkpoints/best.pt` already exists for either run, that run is skipped and only missing evaluation/post-processing is refreshed.

Do **not** run seed 3409 here unless you intentionally choose the expensive n=3 plan later.


In [ ]:
# Colab continuation run: only the remaining ISIC2018 seed-3408 jobs.
# This avoids accidentally launching the broad Option-A sweep after Kaggle outputs
# have already covered Kvasir/CVC and ISIC seed 3407.

import os
import signal
import subprocess
import sys
from pathlib import Path

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

COMMON_OVERRIDES = [
    "train.epochs=150",
    "train.batch_size=8",
    "train.grad_accum_steps=2",
    "train.encoder_lr=0.00005",
    "train.decoder_lr=0.0005",
    "train.nonfinite_grad_action=skip",
    "train.max_nonfinite_grad_steps=10",
    "dataset.split_strategy=pre_split_dirs",
    "dataset.num_workers=2",
    "dataset.prefetch_factor=2",
    "train.save_every=5",
    "runtime.device=cuda",
    "runtime.wandb.mode=offline",
    "evaluation.max_visualizations=8",
]

REMAINING_JOBS = [
    {
        "label": "ISIC2018 A1 baseline seed 3408",
        "config": "configs/isic2018_unet_baseline.yaml",
        "experiment": "isic2018_unet_baseline",
        "run_name": "isic2018_unet_baseline_seed3408",
        "seed": 3408,
    },
    {
        "label": "ISIC2018 A2 full WBSNet seed 3408",
        "config": "configs/isic2018_wbsnet.yaml",
        "experiment": "isic2018_wbsnet",
        "run_name": "isic2018_wbsnet_seed3408",
        "seed": 3408,
    },
]


def run_checked(cmd):
    print("\n>>> " + " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(x) for x in cmd], check=True)


def best_checkpoint(job):
    return Path("outputs") / job["experiment"] / job["run_name"] / "checkpoints" / "best.pt"


def evaluation_json(job):
    return Path("outputs") / job["experiment"] / job["run_name"] / "evaluation" / "ISIC2018_test.json"


sync_cmd = (
    "while true; do "
    f"  rsync -a outputs/ {DRIVE_BACKUP}/ >> /content/rsync.log 2>&1; "
    "  sleep 600; "
    "done"
)
sync_proc = subprocess.Popen(["bash", "-c", sync_cmd], start_new_session=True)
print(f"Started periodic Drive sync (PID {sync_proc.pid}); logs at /content/rsync.log")

try:
    for job in REMAINING_JOBS:
        ckpt = best_checkpoint(job)
        if ckpt.exists():
            print(f"\n[skip train] {job['label']} already has {ckpt}")
        else:
            train_cmd = [
                sys.executable,
                "train.py",
                "--config",
                job["config"],
                "--override",
                f"experiment.seed={job['seed']}",
                *COMMON_OVERRIDES,
                f"experiment.run_name={job['run_name']}",
                "--no-auto-resume",
            ]
            run_checked(train_cmd)

        ckpt = best_checkpoint(job)
        if not ckpt.exists():
            raise FileNotFoundError(f"Expected checkpoint missing after training: {ckpt}")

        metrics_file = evaluation_json(job)
        if metrics_file.exists():
            print(f"[skip eval] {metrics_file} already exists")
        else:
            eval_cmd = [
                sys.executable,
                "evaluate.py",
                "--config",
                job["config"],
                "--checkpoint",
                ckpt,
                "--split",
                "test",
                "--override",
                f"experiment.seed={job['seed']}",
                *COMMON_OVERRIDES,
                f"experiment.run_name={job['run_name']}",
            ]
            run_checked(eval_cmd)

    # Refresh Kvasir -> CVC-ColonDB generalization for whichever imported Kvasir
    # WBSNet seed checkpoints are available. Missing seeds are reported, not fatal.
    for seed in (3407, 3408):
        run_name = f"kvasir_wbsnet_seed{seed}"
        ckpt = Path("outputs/kvasir_wbsnet") / run_name / "checkpoints/best.pt"
        metrics_file = Path("outputs/kvasir_wbsnet") / run_name / "evaluation/CVC-ColonDB_all.json"
        if not ckpt.exists():
            print(f"[skip generalization] missing {ckpt}")
            continue
        if metrics_file.exists():
            print(f"[skip generalization] {metrics_file} already exists")
            continue
        run_checked([
            sys.executable,
            "evaluate.py",
            "--config",
            "configs/kvasir_colondb_generalization.yaml",
            "--checkpoint",
            ckpt,
            "--split",
            "all",
            "--override",
            f"experiment.seed={seed}",
            *COMMON_OVERRIDES,
            f"experiment.run_name={run_name}",
        ])

    run_checked([sys.executable, "aggregate_results.py", "--root", "outputs", "--output", "outputs/aggregated"])
    run_checked([
        sys.executable,
        "scripts/significance_tests.py",
        "--root",
        "outputs",
        "--output",
        "outputs/significance",
        "--record-type",
        "evaluation",
        "--reference",
        "A1_identity_unet",
    ])
    run_checked([sys.executable, "scripts/model_complexity.py", "--output", "outputs/model_complexity"])
finally:
    try:
        os.killpg(os.getpgid(sync_proc.pid), signal.SIGTERM)
        print("Stopped periodic Drive sync.")
    except ProcessLookupError:
        pass
    print("Final rsync to Drive...")
    run_checked(["rsync", "-a", "--info=progress2", "outputs/", f"{DRIVE_BACKUP}/"])


## 10. Generate qualitative paper figures

After training, build the qualitative comparison panels for the paper.

In [ ]:
from pathlib import Path
import subprocess
import sys

candidates = sorted(
    Path("outputs").rglob("checkpoints/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
kvasir_runs = [p for p in candidates if "kvasir_wbsnet" in str(p) and "baseline" not in str(p)]
if not kvasir_runs:
    raise RuntimeError("No Kvasir WBSNet checkpoint found.")

BEST_CKPT = kvasir_runs[0]
print("Using checkpoint:", BEST_CKPT)
PRED_DIR = BEST_CKPT.parent.parent / "predictions"

# The processed Kvasir card may not include a test split, so use val for qualitative panels.
if not PRED_DIR.exists() or not any(PRED_DIR.iterdir()):
    subprocess.run([
        sys.executable, "predict.py",
        "--config", "configs/kvasir_wbsnet.yaml",
        "--checkpoint", str(BEST_CKPT),
        "--split", "val",
        "--override",
        "dataset.split_strategy=pre_split_dirs",
        "dataset.num_workers=2",
        "runtime.wandb.mode=offline",
        "evaluation.max_visualizations=8",
    ], check=True)

subprocess.run([
    sys.executable, "scripts/make_paper_figures.py",
    "--input-dir", str(PRED_DIR),
    "--limit", "8",
    "--columns", "2",
], check=True)


## 11. Save outputs back to Drive

Colab disks reset on disconnect. Always rsync after any major run.

In [ ]:
from pathlib import Path
import subprocess

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", "outputs/", f"{DRIVE_BACKUP}/"], check=True)
print(f"Backed up outputs to {DRIVE_BACKUP}")


## 12. Build the paper PDF (optional, can also be done locally)

In [ ]:
# Uncomment to install LaTeX and build the PDF inside Colab.
# !apt-get install -y --quiet texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-publishers
# !cd paper && pdflatex paper && bibtex paper && pdflatex paper && pdflatex paper